In [58]:
import os
import requests


In [59]:
model = "openai/gpt-3.5-turbo"
url = "https://openrouter.ai/api/v1/chat/completions"

In [60]:
def call_llm(system_prompt , user_prompt, temperature=0.0, max_token = 512):
    api_key = os.environ.get("API_KEY")
    if api_key is None:
        print (f"API key not found")
        return None

    payload = {
        "model":model,
        "messages" : [
            {
                "role" : "system",
                "content" : system_prompt
            },
            {
                "role":"user",
                "content":user_prompt
            }
        ],
        "max_tokens" : max_token,
        "temperature": temperature
    }

    headers = {
        "Authorization":f"Bearer {api_key}",
        "Content-Type":"application/json"
    }
    
    try:
        response = requests.post(url, headers = headers, json=payload)
    except requests.exceptions.ConnectionError as err:
        print(f"Connection Error: {err}")
        return None
    except requests.exceptions.Timeout as err:
        print(f"Request Timed Out: {err}")
        return None
    except requests.exceptions.RequestException as err:
        print(f"Request Failed: {err}")
        return None

    if response.status_code != 200:
        print(f"Error : {response.status_code}")
        print(response.text)
        return None
    try:
        return response.json()["choices"][0]["message"]["content"]
    except (KeyError, IndexError, ValueError):
        print("Unexpected API response:")
        print(response.json())
    return None

In [61]:
system_prompt = "You are a helpful assistant."
user_prompt = "Reply with only the word: hello"

response = call_llm(system_prompt, user_prompt)

print(response)

Hello


In [62]:
#Define SYSTEM PROMPT

system_prompt = """
   You are a healthcare risk assessment assistant.

    Your task is to assess a patient's stroke risk based on the provided health record and return a structured JSON assessment.
    
    Rules:
    1. Do not diagnose diseases.
    2. Do not prescribe medications.
    3. Do not claim that the patient will definitely have a stroke.
    4. Assess the patient's overall risk based on age, hypertension, heart disease, average glucose level, BMI, and smoking status.
    5. Return ONLY valid JSON.
    6. The JSON must contain EXACTLY these five fields:
       - risk_tier
       - flag_for_review
       - primary_signal
       - confidence
       - recommended_action
    7. Do not rename any key.
    8. Do not include any additional fields.

        Example:

        input :
        { 
            "age" : 68,
            "hypertension":1,
            "heart_disease":0,
            "avg_glucose_level":180,
            "bmi":31,
            "smoking_status":"formerly smoked"
        }
        
        output:
       {
            "risk_tier":"Low|Medium|High",
            "flag_for_review":true,
            "primary_signal":"High glucose level and hypertension",
            "confidence":"High",
            "recommended_action":"Consult a healthcare professional for further evaluation and continue monitoring blood pressure and blood glucose levels."
        }
        Do not rename any key in the output, do not include additional fields
        """


user_prompt = """
    Assess the following patient record and return ONLY valid JSON.

    Patient Profile

    Age: 65
    Hypertension: 1
    Heart Disease: 0
    Average Glucose Level: 180
    BMI: 31
    Smoking Status: formerly smoked

"""


response = call_llm(system_prompt, user_prompt, temperature=0)
print(f"Temperature : 0 ")
print(f"Response : {response}")

print()
response2 = call_llm(system_prompt, user_prompt, temperature=0.7)
print(f"Temperature : 0.7")
print(f"Response : {response2}")

Temperature : 0 
Response : {
    "risk_tier": "Medium",
    "flag_for_review": true,
    "primary_signal": "High glucose level and hypertension",
    "confidence": "High",
    "recommended_action": "Consult a healthcare professional for further evaluation and continue monitoring blood pressure and blood glucose levels."
}

Temperature : 0.7
Response : {
    "risk_tier": "Medium",
    "flag_for_review": true,
    "primary_signal": "High glucose level and hypertension",
    "confidence": "High",
    "recommended_action": "Consult a healthcare professional for further evaluation and continue monitoring blood pressure and blood glucose levels."
}


In [63]:
# 3 differnet prompts

print(f"Prompt :1")
patient1 = """
Age:45
Hypertension:0
Heart Disease:0
Average Glucose Level:95
BMI:23
Smoking Status:never smoked
"""
response = call_llm(system_prompt, patient1, temperature=0)
print(f"Temperature : 0 ")
print(f"Response : {response}")
print()

response2 = call_llm(system_prompt, patient1, temperature=0.7)
print(f"Temperature : 0.7 ")

print(f"Response : {response2}")
print("="*20)

patient2 = """
Age:65
Hypertension:1
Heart Disease:0
Average Glucose Level:180
BMI:31
Smoking Status:formerly smoked
"""
print(f"Prompt :2")
response = call_llm(system_prompt, patient2, temperature=0)
print(f"Temperature : 0 ")
print(f"Response : {response}")
print()

response2 = call_llm(system_prompt, patient2, temperature=0.7)
print(f"Temperature : 0.7")
print(f"Response : {response2}")
print("="*20)
print()
patient3 = """
Age:72
Hypertension:1
Heart Disease:1
Average Glucose Level:210
BMI:34
Smoking Status:smokes
"""

print(f"Prompt : 3")
response = call_llm(system_prompt, patient3, temperature=0)
print(f"Temperature : 0 ")

print(f"Response : {response}")
print()

response2 = call_llm(system_prompt, patient3, temperature=0.7)
print(f"Temperature : 0.7 ")

print(f"Response : {response2}")

Prompt :1
Temperature : 0 
Response : {
    "risk_tier": "Low",
    "flag_for_review": false,
    "primary_signal": "No significant risk factors",
    "confidence": "Low",
    "recommended_action": "Maintain a healthy lifestyle and continue regular check-ups with a healthcare provider."
}

Temperature : 0.7 
Response : {
    "risk_tier": "Low",
    "flag_for_review": false,
    "primary_signal": "No significant risk factors",
    "confidence": "Low",
    "recommended_action": "Maintain a healthy lifestyle and monitor blood pressure and blood glucose levels regularly."
}
Prompt :2
Temperature : 0 
Response : {
    "risk_tier": "Medium",
    "flag_for_review": true,
    "primary_signal": "High glucose level and hypertension",
    "confidence": "High",
    "recommended_action": "Consult a healthcare professional for further evaluation and continue monitoring blood pressure and blood glucose levels."
}

Temperature : 0.7
Response : {
    "risk_tier": "Medium",
    "flag_for_review": true,


In [70]:
from jsonschema import validate

schema_format= {
    "type": "object",
    "properties": {
        "risk_tier": {
            "type": "string",
            "enum": ["Low", "Medium", "High"]
        },
        "flag_for_review": {
            "type": "boolean"
        },
        "primary_signal": {
            "type": "string"
        },
        "confidence": {
            "type": "string",
            "enum": ["Low", "Medium", "High"]
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "risk_tier",
        "flag_for_review",
        "primary_signal",
        "confidence",
        "recommended_action"
    ],
    "additionalProperties": False
}

In [71]:
fallback = {
    "risk_tier": None,
    "flag_for_review": None,
    "primary_signal": None,
    "confidence": None,
    "recommended_action": None
}


#Prepare three dataset records
records = [
    {
        "age": 45,
        "hypertension": 0,
        "heart_disease": 0,
        "avg_glucose_level": 95,
        "bmi": 23,
        "smoking_status": "never smoked"
    },
    {
        "age": 65,
        "hypertension": 1,
        "heart_disease": 0,
        "avg_glucose_level": 180,
        "bmi": 31,
        "smoking_status": "formerly smoked"
    },
    {
        "age": 72,
        "hypertension": 1,
        "heart_disease": 1,
        "avg_glucose_level": 210,
        "bmi": 34,
        "smoking_status": "smokes"
    },
    {
        "age": 29,
        "hypertension": 1,
        "heart_disease": 0,
        "avg_glucose_level": 100,
        "bmi": 24,
        "smoking_status": "never smoked",
        "email" : "Idnhu@gmail.com"
    },
]

In [72]:

#Call LLM and validate


import json
from jsonschema import validate, ValidationError

for i, record in enumerate(records, start=1):

    user_prompt = json.dumps(record, indent=2)

    raw_response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print(f"\nRecord {i}")
    print("Input:")
    print(user_prompt)
    print("\nRaw LLM Response:")
    print(raw_response)

    try:
        response_json = json.loads(raw_response.strip())

        validate(
            instance=response_json,
            schema=schema_format
        )

        print("Validation: SUCCESS")

    except json.JSONDecodeError as err:

        print("JSON Parsing Error:", err)

        response_json = fallback

        print("Validation: FAILED")

    except ValidationError as err:

        print("Schema Validation Error:", err.message)

        response_json = fallback

        print("Validation: FAILED")

    print("\nFinal Output")
    print(response_json)


Record 1
Input:
{
  "age": 45,
  "hypertension": 0,
  "heart_disease": 0,
  "avg_glucose_level": 95,
  "bmi": 23,
  "smoking_status": "never smoked"
}

Raw LLM Response:
{
    "risk_tier": "Low",
    "flag_for_review": false,
    "primary_signal": "No significant risk factors detected",
    "confidence": "Low",
    "recommended_action": "Maintain a healthy lifestyle and attend regular check-ups."
}
Validation: SUCCESS

Final Output
{'risk_tier': 'Low', 'flag_for_review': False, 'primary_signal': 'No significant risk factors detected', 'confidence': 'Low', 'recommended_action': 'Maintain a healthy lifestyle and attend regular check-ups.'}

Record 2
Input:
{
  "age": 65,
  "hypertension": 1,
  "heart_disease": 0,
  "avg_glucose_level": 180,
  "bmi": 31,
  "smoking_status": "formerly smoked"
}

Raw LLM Response:
{
    "risk_tier": "Medium",
    "flag_for_review": true,
    "primary_signal": "High glucose level and hypertension",
    "confidence": "High",
    "recommended_action": "Consul

In [73]:
#PII detection function

import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

#wrapper method for call_llm

def safe_call_llm(system_prompt,
                  user_prompt,
                  temperature=0.0,
                  max_tokens=512):

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    return call_llm(
        system_prompt,
        user_prompt,
        temperature,
        max_tokens
    )

#guardrail : Test1 should be blocked
print(f"Test case 1 : Should be Blocked")
user_prompt = """
Patient Name: John

Email: john.doe@gmail.com

Age: 65
BMI: 31
"""

response = safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)
print()
print(f"Test case-II should be passed")
#guardrail : TEst2 should be passed
user_prompt = """
Age: 65

Hypertension: 1

Heart Disease: 0

Average Glucose Level: 180

BMI: 31

Smoking Status: formerly smoked
"""

response = safe_call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

print(response)

Test case 1 : Should be Blocked
Input blocked: PII detected.
None

Test case-II should be passed
{
    "risk_tier": "High",
    "flag_for_review": true,
    "primary_signal": "High glucose level and hypertension",
    "confidence": "High",
    "recommended_action": "Consult a healthcare professional for further evaluation and continue monitoring blood pressure and blood glucose levels."
}


In [75]:

#Call safe LLM and validate, quardrails for 4 sample input


import json
from jsonschema import validate, ValidationError

for i, record in enumerate(records, start=1):

    user_prompt = json.dumps(record, indent=2)

    raw_response = safe_call_llm(
        system_prompt,
        user_prompt,
        temperature=0
    )

    print(f"\nRecord {i}")
    print("Input:")
    print(user_prompt)

    if raw_response is None:
        print(f"\nGuardrail Result : BLOCKED , PII detected")
        print(f"JSON Validation :Not Performed")
        continue

    print("\nGuardrail Result: PASSED")
    print("\nRaw LLM Response:")
    print(raw_response)
    

    try:
        response_json = json.loads(raw_response.strip())

        validate(
            instance=response_json,
            schema=schema_format
        )

        print("Validation: SUCCESS")

    except json.JSONDecodeError as err:

        print("JSON Parsing Error:", err)

        response_json = fallback

        print("Validation: FAILED")

    except ValidationError as err:

        print("Schema Validation Error:", err.message)

        response_json = fallback

        print("Validation: FAILED")

    print("\nFinal Output")
    print(response_json)
    print("="*40)


Record 1
Input:
{
  "age": 45,
  "hypertension": 0,
  "heart_disease": 0,
  "avg_glucose_level": 95,
  "bmi": 23,
  "smoking_status": "never smoked"
}

Guardrail Result: PASSED

Raw LLM Response:
{
    "risk_tier": "Low",
    "flag_for_review": false,
    "primary_signal": "No significant risk factors detected",
    "confidence": "Low",
    "recommended_action": "Maintain a healthy lifestyle and attend regular check-ups."
}
Validation: SUCCESS

Final Output
{'risk_tier': 'Low', 'flag_for_review': False, 'primary_signal': 'No significant risk factors detected', 'confidence': 'Low', 'recommended_action': 'Maintain a healthy lifestyle and attend regular check-ups.'}

Record 2
Input:
{
  "age": 65,
  "hypertension": 1,
  "heart_disease": 0,
  "avg_glucose_level": 180,
  "bmi": 31,
  "smoking_status": "formerly smoked"
}

Guardrail Result: PASSED

Raw LLM Response:
{
    "risk_tier": "Medium",
    "flag_for_review": true,
    "primary_signal": "High glucose level and hypertension",
    "co